In [33]:
import os
import librosa
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# --- 1. Konfiguration nach Nature Paper & S1000R Anforderungen ---
TARGET_SR = 48000
SEGMENT_DURATION = 1.0  
IMAGE_SIZE = (224, 224) 
N_FFT = 2048
HOP_LENGTH = 1024
FMIN = 1000       
FMAX = 10000
OVERLAP = 0.75  # 75% Überlappung
STEP_DURATION = SEGMENT_DURATION * (1 - OVERLAP) # Springe nur 0.25s weiter

# --- 2. DEINE LABEL-DEFINITION ---
# Hier definierst du manuell, wo der Leerlauf in deinen Aufnahmen ist
# Format: 'filename': (start_sekunde, end_sekunde)
idle_segments = [
    ('S1000R_2015_PhilipKehl.m4a', 0.5, 10.5),
    ('S1000R_2015_PhilipKehl.m4a', 18, 19),
    ('S1000R_2015_PhilipKehl_cold.m4a', 4.5, 14.5),
    ('S1000R_2015_PhilipKehl_cold.m4a', 20, 24),
    #('S1000R_2015_PhilipKehl_left.m4a', 0.5, 4.5),
    #('S1000R_2015_PhilipKehl_left.m4a', 9.0, 10.0),
    ('S1000R_2017_AndreRobert.m4a', 4.5, 15.5),
    ('S1000RR_2014_MaximilianHohmann.m4a', 0.5, 20.0),
    ('S1000RR_2014_MaximilianHohmann.m4a', 40.0, 47.0),
    ('S1000R_2014_YoutubeVToldsMotoShow.m4a', 0.0, 5.0),
    ('S1000RR_2012_YoutubeMartinNgim.m4a', 0.0, 10.0)
]

# Ordnerstruktur erstellen
output_dir = '../data/processed/dataset_idle'
os.makedirs(output_dir, exist_ok=True)

def generate_spectrogram_image(y_segment, sr, output_path):
    # STFT berechnen
    S = np.abs(librosa.stft(y_segment, n_fft=N_FFT, hop_length=HOP_LENGTH))
    
    # Normalisierung: Der lauteste Peak im Segment wird 0 dB
    # Das eliminiert Handy-Bewegungen und AGC-Effekte
    S_db = librosa.amplitude_to_db(S, ref=np.max)
    
    # Manuelles Cropping auf Frequenzen im Bereich FMIN bis FMAX
    freqs = librosa.fft_frequencies(sr=sr, n_fft=N_FFT)
    idx_fmin = np.argmin(np.abs(freqs - FMIN))
    idx_fmax = np.argmin(np.abs(freqs - FMAX))
    S_db_cropped = S_db[idx_fmin:idx_fmax, :]
    
    # Bild-Erzeugung in 224x224 (ohne Ränder)
    fig = plt.figure(figsize=(2.24, 2.24), dpi=100)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.axis('off')
    
    # Dynamikumfang fixieren: 80dB Kontrast
    ax.imshow(S_db_cropped, aspect='auto', origin='lower', cmap='magma', vmin=-80, vmax=0)
    
    plt.savefig(output_path, pad_inches=0)
    plt.close(fig)
    
    # Resize mit hoher Qualität (LANCZOS)
    img = Image.open(output_path).convert('RGB').resize(IMAGE_SIZE, Image.Resampling.LANCZOS)
    img.save(output_path)

# --- 2. Pipeline-Ausführung ---
for file_name, start_sec, end_sec in idle_segments:
    file_path = f'../data/raw/{file_name}'
    y, sr = librosa.load(file_path, sr=TARGET_SR, offset=start_sec, duration=(end_sec - start_sec))
    
    samples_per_segment = int(SEGMENT_DURATION * sr)
    samples_per_step = int(STEP_DURATION * sr) # Der "Sprung" zum nächsten Fenster
    
    # Wir berechnen die Anzahl der möglichen Fenster mit Sliding Window
    num_windows = int((len(y) - samples_per_segment) / samples_per_step) + 1
    
    print(f"Generiere {num_windows} Bilder (mit {int(OVERLAP*100)}% Overlap) für {file_name}...")
    
    for i in range(num_windows):
        start_sample = i * samples_per_step
        end_sample = start_sample + samples_per_segment
        segment_audio = y[start_sample:end_sample]
        
        # Dateiname bekommt jetzt den Zeitstempel für Eindeutigkeit
        timestamp = (start_sec + (i * STEP_DURATION))
        img_name = f"{file_name.split('.')[0]}_idle_{timestamp:.2f}s.png"
        img_path = os.path.join(output_dir, img_name)
        
        generate_spectrogram_image(segment_audio, sr, img_path)

print(f"\nErfolg! {len(os.listdir(output_dir))} Bilder wurden in '{output_dir}' gespeichert.")

/var/folders/9w/hykx30j92h11_5dq0nb94yth0000gn/T/ipykernel_2664/1250041752.py:71: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(file_path, sr=TARGET_SR, offset=start_sec, duration=(end_sec - start_sec))


Generiere 37 Bilder (mit 75% Overlap) für S1000R_2015_PhilipKehl.m4a...
Generiere 1 Bilder (mit 75% Overlap) für S1000R_2015_PhilipKehl.m4a...
Generiere 37 Bilder (mit 75% Overlap) für S1000R_2015_PhilipKehl_cold.m4a...
Generiere 13 Bilder (mit 75% Overlap) für S1000R_2015_PhilipKehl_cold.m4a...
Generiere 41 Bilder (mit 75% Overlap) für S1000R_2017_AndreRobert.m4a...
Generiere 75 Bilder (mit 75% Overlap) für S1000RR_2014_MaximilianHohmann.m4a...
Generiere 25 Bilder (mit 75% Overlap) für S1000RR_2014_MaximilianHohmann.m4a...
Generiere 17 Bilder (mit 75% Overlap) für S1000R_2014_YoutubeVToldsMotoShow.m4a...
Generiere 37 Bilder (mit 75% Overlap) für S1000RR_2012_YoutubeMartinNgim.m4a...

Erfolg! 283 Bilder wurden in '../data/processed/dataset_idle' gespeichert.
